
# **Homework: Building PPO from Scratch**
**Course:** Applied Reinforcement Learning Systems  
**Week:** 2  
**Due:** Next Monday at 11:59 PM  
**Points:** 100  

---

### 🧠 Overview

In this assignment, you will implement **Proximal Policy Optimization (PPO)** from scratch using PyTorch and Gymnasium — without relying on external RL libraries such as Stable Baselines3.

Your goals:
- Build the core PPO training algorithm.
- Implement Generalized Advantage Estimation (GAE).
- Train your agent on `CartPole-v1` or `MountainCarContinuous-v0`.
- Plot reward and diagnostic curves.

---


## 1. Environment Setup (10 pts)

In [7]:

# Install dependencies if needed
# !pip install torch gymnasium matplotlib numpy

import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

env = gym.make("CartPole-v1")
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n

print(f"Observation dim: {obs_dim}, Action dim: {act_dim}")


Observation dim: 4, Action dim: 2


## 2. Policy and Value Networks (15 pts)

In [ ]:
NEURON_NODE_NUM = 64

class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        # NOTE: It is usually good practice to flatten the inputs;
        # However, our observation and action space are linear already
        self.simple_network = nn.Sequential(
            nn.Linear(obs_dim, NEURON_NODE_NUM),
            nn.ReLU(),
            nn.Linear(NEURON_NODE_NUM, NEURON_NODE_NUM),
            nn.ReLU(),
            nn.Linear(NEURON_NODE_NUM, act_dim))

    def forward(self, x):
        logits = self.simple_network(x)
        return torch.distributions.Categorical(logits=logits)

class Critic(nn.Module):
    def __init__(self, obs_dim):
        super().__init__()
        self.simple_network = nn.Sequential(
            nn.Linear(obs_dim, NEURON_NODE_NUM),
            nn.ReLU(),
            nn.Linear(NEURON_NODE_NUM, NEURON_NODE_NUM),
            nn.ReLU(),
            nn.Linear(NEURON_NODE_NUM, 1)) # We want a single value to evaluate from
    
    def forward(self, x):
        return self.simple_network(x).squeeze(-1)
    
actor_nn  = Actor(obs_dim, act_dim).to(DEVICE)
critic_nn = Critic(obs_dim).to(DEVICE)

## 3. Rollout Collection (15 pts)

In [ ]:

# TODO: Implement function to collect transitions (s, a, r, done, value, logp)

def collect_rollout(env, actor, critic, batch_size, gamma=0.99):

    obs  = np.zeros_like(batch_size, dtype=np.float32)
    act  = np.zeros_like(batch_size, dtype=np.float32)
    rew  = np.zeros_like(batch_size, dtype=np.float32)
    done = np.zeros_like(batch_size, dtype=np.float32)
    val  = np.zeros_like(batch_size, dtype=np.float32)
    logp = np.zeros_like(batch_size, dtype=np.float32)

    observation = env.reset()
    for _ in range(batch_size):
        obs_tensor = torch.tensor(observation, dtype=torch.float32, device=DEVICE) # Use tensors for device efficiency
        
        # Iterate a step using networks
        dist  = actor.forward(obs_tensor)
        value = critic.forward(obs_tensor)

        # Collect info from distribution
        action   = dist.sample()
        log_prob = dist.log_prob()

        # Take action
        observation, reward, terminated, truncated, _ = env.step(action.item())
        complete = torch.tensor(terminated or truncated, dtype=torch.BoolType, device=DEVICE)

        # Collect rollout fields
        obs.append(observation)
        act.append(action)
        rew.append(reward)
        done.append(complete)
        val.append(value)
        logp.append(log_prob)

    # Save to dict
    rollout = {
        'obs'  : obs,
        'act'  : act,
        'rew'  : rew,
        'done' : done,
        'val'  : val,
        'logp' : log_prob
    }

    return rollout

# Example expected keys in returned dictionary:
# {'obs': np.array, 'act': np.array, 'rew': np.array, 'done': np.array,
#  'val': np.array, 'logp': np.array}


## 4. Compute GAE and Returns (15 pts)

In [ ]:
def compute_gae(rews, vals, dones, gamma=0.99, lam=0.95):
    advantages     = np.zeros_like(rews, dtype=np.float32)
    last_advantage = 0.0

    if len(rews) != len(vals):
        return # needs to be the same length
    
    # Enumerate backwards
    for t, r_t  in reversed(list(enumerate(rews))): 

        if dones[t]: # Terminated episode
            last_advantage = 0.0 

        sigma_t = r_t + gamma * vals[t+1] - vals[t] if t < len(rews) - 1 else rews[t] - vals[t]
        last_advantage = sigma_t + gamma * lam * last_advantage
        advantages[t] = last_advantage

    return advantages

# Output: advantages and returns

## 5. PPO Update (25 pts)

In [ ]:

# TODO: Implement PPO loss and update step

def ppo_update(actor, critic, optimizer_actor, optimizer_critic, rollout, clip_eps=0.2, epochs=10):
    
    for t in range(epochs):
        ratio  = torch.exp(new_logp - old_logp)

        surr_1 = ratio * advantages

        clipped_ratios = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps)
        surr_2 = clipped_ratios * advantages

        clipped_loss = torch.min(surr_1, surr_2)


## 6. Training Loop (15 pts)

In [ ]:

# TODO: Combine everything into a PPO training loop

# Hints:
# 1. Collect rollouts
# 2. Compute GAE advantages and returns
# 3. Run PPO update
# 4. Print or log results

# Optional: store mean rewards for plotting

rewards_history = []
BATCH_SIZE = 250
critic_optim = nn.MSELoss()
actor_optim  = torch.optim.Adam()


for iteration in range(200):
    rollout    = collect_rollout(env, actor_nn, critic_nn, BATCH_SIZE)
    advantages = compute_gae(rollout['rews'], rollout['vals'], rollout['dones'])
    ppo_update(actor_nn, critic_nn, actor_optim, critic_optim, rollout)
    
    

# Plot rewards
plt.plot(rewards_history)
plt.xlabel("Iteration")
plt.ylabel("Mean reward")
plt.title("PPO from scratch — CartPole-v1")
plt.grid(True)
plt.show()


## 7. Diagnostics and Plots (10 pts)

In [ ]:

# TODO: Add plots for:
# - Advantage distribution
# - Ratio histogram (exp(new_logp - old_logp))

# Example:
# plt.hist(advantages, bins=20)
# plt.title("Advantage distribution")
# plt.show()



## ✍️ Write-Up (10 pts)

Answer the following in a Markdown cell or PDF:
1. What was the most challenging part to implement?
2. How did the clip range and advantage normalization affect learning?
3. What metrics or plots helped you understand PPO's behavior?

---

### 🧩 Extra Credit (+10 pts)
- Add entropy regularization to the policy loss.
- Try `clip_eps=0.1` and `clip_eps=0.3` and compare results.
- Train on `LunarLander-v3` instead of CartPole (requires Box2D).

---

### 💯 Grading Rubric

| Component | Points |
|------------|--------|
| Environment setup and architecture | 10 |
| Rollout buffer | 15 |
| Advantage (GAE) computation | 15 |
| PPO loss and update | 25 |
| Training loop and results | 15 |
| Plots and interpretation | 10 |
| Write-up clarity | 10 |
| **Total** | **100** |

Good luck — remember, the goal is to *understand* PPO, not just make it work!
